In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA SILVER
gold_silver = spark.table("spotify_catalog.silver.spotify_tracks")

display(gold_silver.limit(10))

In [0]:
dim_track_base = (
    gold_silver
    .select(
        "track_id",
        "track_name",
        "duration_seconds",
        "explicit",
        "popularity",
        "popularity_level",
        "album_id"
    )
    .filter(col("track_id").isNotNull())
    .dropDuplicates(["track_id"])
)

In [0]:
window_track = Window.orderBy("track_id")

dim_track = (
    dim_track_base
    .withColumn(
        "sk_track",
        row_number().over(window_track)
    )
    .select(
        "sk_track",
        "track_id",
        "track_name",
        "duration_seconds",
        "explicit",
        "popularity",
        "popularity_level",
        "album_id"
    )
)

In [0]:
(
    dim_track
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_track"
    )
)